In [8]:
!pip install -q groq sentence-transformers scikit-learn tqdm

import os, json, time, re, hashlib, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from groq import Groq
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

# Configuración

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, shutil, json
import pandas as pd
from sklearn.metrics import roc_auc_score
os.chdir('/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/notebooks')

BACKEND = "groq"
MODEL = "llama-3.3-70b-versatile"
# MODEL = "llama-3.1-8b-instant"
METHOD = "simmia"
LENGTH = "medium"

N_SAMPLES = 3       # muestras por prefijo
MAX_WORDS = 20      # palabras del texto usadas
MAX_NEW_TOKENS = 3  # queremos que genere próxima palabra / mini continuación
SLEEP_BETWEEN_CALLS = 2.1
SEED = 2319

BASE_DIR = "/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026"
DATASET_PATH = "/content/drive/MyDrive/a udesa/7. septimo semestre/nlp/TP NLP 2026/datasets/evaluation/booktection_medium_eval_20books_10chunks.csv"

API_LOG_DIR = os.path.join(BASE_DIR, "api_logs", METHOD, MODEL.replace("/", "_"))
RESULTS_DIR = os.path.join(BASE_DIR, "results", METHOD, MODEL.replace("/", "_"))

os.makedirs(API_LOG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

try:
    from google.colab import userdata

    api_keys = []

    for i in range(1, 12):
        name = "GROQ_API_KEY" if i == 1 else f"GROQ_API_KEY{i}"
        try:
            k = userdata.get(name)
            if k:
                api_keys.append(k)
        except:
            pass

except Exception:
    api_keys = []

print(f"Se cargaron {len(api_keys)} API Keys")

current_key = 0
client = Groq(api_key=api_keys[current_key])

HF_TOKEN = userdata.get("HGF")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Se cargaron 11 API Keys


# Cargar dataset

In [10]:
df = pd.read_csv(DATASET_PATH)
df = df[df["length"] == LENGTH].reset_index(drop=True)

print(df.shape)
display(df.head(2))

(200, 10)


,sample_id,book_id,label,length,answer,text,Example_A,Example_B,Example_C,Example_D
0,BT_0000,1984_-_George_Orwell,1,medium,A,Since he was arrested he had not been fed. He ...,Since he was arrested he had not been fed. He ...,He had not eaten anything since being arrested...,"Since his arrest, he had been given nothing to...",He had been given no food since being arrested...
1,BT_0001,The_Alchemist_-_Paulo_Coelho,1,medium,A,"“And I know the Soul of the World, because we ...","“And I know the Soul of the World, because we ...",The essence of the universe has confided in me...,I'm privy to the psyche of the cosmos from our...,The spirit of the world has confided in me dur...


# Backend del modelo


In [11]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    token=HF_TOKEN
)

def similarity(w1, w2):
    if not isinstance(w1, str) or not isinstance(w2, str):
        return 0.0
    if not w1.strip() or not w2.strip():
        return 0.0

    e1 = embedder.encode([w1])
    e2 = embedder.encode([w2])
    cos = cosine_similarity(e1, e2)[0][0]
    return float((cos + 1) / 2)

def first_word(text):
    if text is None:
        return ""
    text = str(text).strip()
    text = re.sub(r"^[^\w]+", "", text)
    parts = text.split()
    return parts[0] if parts else ""

def switch_api_key():
    global current_key, client

    if current_key + 1 >= len(api_keys):
        raise DailyCapError("Todas las API keys agotadas.")

    current_key += 1
    client = Groq(api_key=api_keys[current_key])

    print(f"\n🔑 Cambiando a API KEY #{current_key+1}")

class DailyCapError(Exception):
    pass

def safe_generate(prefix, max_retries=6):
    for attempt in range(max_retries):
        try:
            r = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "Continue the text naturally. Output only the next word or very short continuation."
                    },
                    {
                        "role": "user",
                        "content": prefix
                    }
                ],
                max_tokens=MAX_NEW_TOKENS,
                temperature=1.0,
                top_p=1.0,
                seed=SEED + attempt
            )
            return r.choices[0].message.content.strip()

        except Exception as e:
            msg = str(e)

            if "429" not in msg and "rate_limit" not in msg.lower():
                raise

            if "per day" in msg.lower() or "TPD" in msg or "RPD" in msg:
                switch_api_key()
                continue

            m2 = re.search(r"try again in (\d+)m([\d.]+)s", msg)
            m1 = re.search(r"try again in ([\d.]+)s", msg)

            wait = (
                int(m2.group(1)) * 60 + float(m2.group(2))
                if m2 else
                float(m1.group(1)) if m1 else 20.0
            )

            time.sleep(wait + 1)

    raise RuntimeError("max_retries agotado")

def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def build_non_member_prefix(df, current_sample_id=None, n_shots=3):
    pool = df[df["label"] == 0].copy()

    if current_sample_id is not None:
        pool = pool[pool["sample_id"] != current_sample_id]

    if len(pool) == 0:
        return ""

    n = min(n_shots, len(pool))

    return " ".join(
        pool.sample(n=n, random_state=SEED)["text"].astype(str).tolist()
    )


def make_query_id(sample_id, position, sample_idx, query_type="normal"):
    raw = f"{METHOD}|{MODEL}|{sample_id}|{position}|{sample_idx}|{query_type}|{MAX_WORDS}|{N_SAMPLES}|{SEED}"
    return hashlib.md5(raw.encode("utf-8")).hexdigest()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# simMIA

In [12]:
sample_csv = os.path.join(RESULTS_DIR, "simmia_sample_level_results.csv")
token_csv = os.path.join(RESULTS_DIR, "simmia_token_level_results.csv")
query_csv = os.path.join(RESULTS_DIR, "simmia_query_level_results.csv")

if os.path.exists(sample_csv):
    previous_sample_df = pd.read_csv(sample_csv).drop_duplicates(subset=["sample_id"])
    completed_sample_ids = set(previous_sample_df["sample_id"].astype(str))
    sample_records = previous_sample_df.to_dict("records")
    print(f"Checkpoint sample-level encontrado: {len(completed_sample_ids)} samples completos")
else:
    completed_sample_ids = set()
    sample_records = []

if os.path.exists(token_csv):
    previous_token_df = pd.read_csv(token_csv).drop_duplicates(subset=["sample_id", "position"])
    token_records = previous_token_df.to_dict("records")
else:
    token_records = []

if os.path.exists(query_csv):
    previous_query_df = pd.read_csv(query_csv).drop_duplicates(subset=["query_id"])
    query_records = previous_query_df.to_dict("records")
else:
    query_records = []

stopped = False
EPS = 1e-8
N_NON_MEMBER_SHOTS = 3

for _, row in tqdm(df.iterrows(), total=len(df), desc="SimMIA samples"):

    sample_id = row["sample_id"]
    if str(sample_id) in completed_sample_ids:
        continue
    book_id = row["book_id"]
    label = int(row["label"])
    text = row["text"]

    words = str(text).split()[:MAX_WORDS]

    if len(words) < 2:
        continue

    token_scores = []
    normal_token_scores = []
    perturbed_token_scores = []

    non_member_prefix = build_non_member_prefix(
        df,
        current_sample_id=sample_id,
        n_shots=N_NON_MEMBER_SHOTS
    )

    try:
        for i in range(1, len(words)):
            prefix = " ".join(words[:i])
            target_word = words[i]

            normal_generated_words = []
            perturbed_generated_words = []
            normal_similarities = []
            perturbed_similarities = []

            for sample_idx in range(N_SAMPLES):

                # -------- NORMAL QUERY: s(x_i | x_<i) --------
                normal_query_id = make_query_id(
                    sample_id, i, sample_idx, query_type="normal"
                )
                normal_log_path = os.path.join(API_LOG_DIR, f"{normal_query_id}.json")

                if os.path.exists(normal_log_path):
                    normal_record = load_json(normal_log_path)
                    normal_generated = normal_record["generated_text"]
                    normal_gen_word = normal_record["generated_first_word"]
                    normal_sim = normal_record["similarity"]

                else:
                    normal_generated = safe_generate(prefix)
                    normal_gen_word = first_word(normal_generated)
                    normal_sim = similarity(target_word, normal_gen_word)

                    normal_record = {
                        "query_id": normal_query_id,
                        "query_type": "normal",
                        "method": METHOD,
                        "model": MODEL,
                        "backend": BACKEND,
                        "sample_id": sample_id,
                        "book_id": book_id,
                        "label": label,
                        "length": LENGTH,
                        "position": i,
                        "prefix": prefix,
                        "target_word": target_word,
                        "sample_idx": sample_idx,
                        "generated_text": normal_generated,
                        "generated_first_word": normal_gen_word,
                        "similarity": normal_sim,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                    }

                    save_json(normal_log_path, normal_record)

                    if SLEEP_BETWEEN_CALLS:
                        time.sleep(SLEEP_BETWEEN_CALLS)

                # -------- PERTURBED QUERY: s(x_i | P ⊕ x_<i) --------
                perturbed_prefix = (non_member_prefix + " " + prefix).strip()

                perturbed_query_id = make_query_id(
                    sample_id, i, sample_idx, query_type="perturbed"
                )
                perturbed_log_path = os.path.join(API_LOG_DIR, f"{perturbed_query_id}.json")

                if os.path.exists(perturbed_log_path):
                    perturbed_record = load_json(perturbed_log_path)
                    perturbed_generated = perturbed_record["generated_text"]
                    perturbed_gen_word = perturbed_record["generated_first_word"]
                    perturbed_sim = perturbed_record["similarity"]

                else:
                    perturbed_generated = safe_generate(perturbed_prefix)
                    perturbed_gen_word = first_word(perturbed_generated)
                    perturbed_sim = similarity(target_word, perturbed_gen_word)

                    perturbed_record = {
                        "query_id": perturbed_query_id,
                        "query_type": "perturbed",
                        "method": METHOD,
                        "model": MODEL,
                        "backend": BACKEND,
                        "sample_id": sample_id,
                        "book_id": book_id,
                        "label": label,
                        "length": LENGTH,
                        "position": i,
                        "prefix": perturbed_prefix,
                        "original_prefix": prefix,
                        "non_member_prefix": non_member_prefix,
                        "target_word": target_word,
                        "sample_idx": sample_idx,
                        "generated_text": perturbed_generated,
                        "generated_first_word": perturbed_gen_word,
                        "similarity": perturbed_sim,
                        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
                    }

                    save_json(perturbed_log_path, perturbed_record)

                    if SLEEP_BETWEEN_CALLS:
                        time.sleep(SLEEP_BETWEEN_CALLS)

                query_records.append(normal_record)
                query_records.append(perturbed_record)

                normal_generated_words.append(normal_gen_word)
                perturbed_generated_words.append(perturbed_gen_word)

                normal_similarities.append(normal_sim)
                perturbed_similarities.append(perturbed_sim)

            normal_word_score = float(np.mean(normal_similarities))
            perturbed_word_score = float(np.mean(perturbed_similarities))
            relative_word_score = float(
                - perturbed_word_score / (normal_word_score + EPS)
            )

            token_scores.append(relative_word_score)
            normal_token_scores.append(normal_word_score)
            perturbed_token_scores.append(perturbed_word_score)

            token_records.append({
                "sample_id": sample_id,
                "book_id": book_id,
                "label": label,
                "position": i,
                "target_word": target_word,
                "normal_word_score": normal_word_score,
                "perturbed_word_score": perturbed_word_score,
                "relative_word_score": relative_word_score,
                "n_samples": N_SAMPLES,
                "normal_generated_words": normal_generated_words,
                "perturbed_generated_words": perturbed_generated_words
            })

        raw_membership_score = float(np.mean(token_scores))
        normal_membership_score = float(np.mean(normal_token_scores))
        perturbed_membership_score = float(np.mean(perturbed_token_scores))

        sample_records.append({
            "sample_id": sample_id,
            "book_id": book_id,
            "label": label,
            "method": METHOD,
            "model": MODEL,
            "backend": BACKEND,
            "raw_membership_score": raw_membership_score,
            "normal_membership_score": normal_membership_score,
            "perturbed_membership_score": perturbed_membership_score,
            "n_positions": len(token_scores),
            "max_words": MAX_WORDS,
            "n_samples_per_prefix": N_SAMPLES,
            "n_non_member_shots": N_NON_MEMBER_SHOTS
        })

        query_df = pd.DataFrame(query_records).drop_duplicates(subset=["query_id"])
        token_df = pd.DataFrame(token_records).drop_duplicates(subset=["sample_id", "position"])
        sample_df = pd.DataFrame(sample_records).drop_duplicates(subset=["sample_id"])

        query_df.to_csv(query_csv, index=False)
        token_df.to_csv(token_csv, index=False)
        sample_df.to_csv(sample_csv, index=False)

        completed_sample_ids.add(str(sample_id))

    except DailyCapError:
        stopped = True
        print("\n⚠️ Cap diario de Groq alcanzado.")
        print("Reanudá después re-corriendo esta celda: las consultas ya guardadas no se repiten.")
        break

Checkpoint sample-level encontrado: 200 samples completos


SimMIA samples:   0%|          | 0/200 [00:00<?, ?it/s]

# Resultados

In [ ]:
query_df = pd.DataFrame(query_records)
token_df = pd.DataFrame(token_records)
sample_df = pd.DataFrame(sample_records)

if len(sample_df) > 0:
    sample_df = sample_df.dropna(subset=["raw_membership_score"]).reset_index(drop=True)

    # Score crudo del paper: usar para AUC
    sample_df["membership_score"] = sample_df["raw_membership_score"]

    # Probabilidad normalizada: usar para MAE y comparar métodos
    min_s = sample_df["raw_membership_score"].min()
    max_s = sample_df["raw_membership_score"].max()

    if max_s > min_s:
        sample_df["membership_probability"] = (
            sample_df["raw_membership_score"] - min_s
        ) / (max_s - min_s)
    else:
        sample_df["membership_probability"] = 0.5

    sample_df["absolute_error"] = abs(
        sample_df["label"] - sample_df["membership_probability"]
    )

query_csv = os.path.join(RESULTS_DIR, "simmia_query_level_results.csv")
token_csv = os.path.join(RESULTS_DIR, "simmia_token_level_results.csv")
sample_csv = os.path.join(RESULTS_DIR, "simmia_sample_level_results.csv")

query_df.to_csv(query_csv, index=False)
token_df.to_csv(token_csv, index=False)
sample_df.to_csv(sample_csv, index=False)

print("CORTADO por cap diario." if stopped else "COMPLETO.")
print("Queries guardadas:", len(query_df))
print("Tokens completos:", len(token_df))
print("Samples completos:", len(sample_df))

print("\nQuery-level:", query_csv)
print("Token-level:", token_csv)
print("Sample-level:", sample_csv)

display(sample_df.head())

In [ ]:
if len(sample_df) > 0:
    print("Modelo:", MODEL)
    print("Método: SimMIA")
    print("Samples completos:", len(sample_df))

    print("\nMean Absolute Error:")
    print(sample_df["absolute_error"].mean())

    print("\nError por clase:")
    print(sample_df.groupby("label")["absolute_error"].mean())

    if sample_df["label"].nunique() == 2:
        y = sample_df["label"].values
        s = sample_df["raw_membership_score"].values

        auc = roc_auc_score(y, s)
        auc_inv = roc_auc_score(y, -s)

        print("\nAUC:", auc)
        print("AUC invertido:", auc_inv)

        fpr, tpr, _ = roc_curve(y, s)

        plt.figure(figsize=(6, 5))
        plt.plot(fpr, tpr, lw=2, label=f"{MODEL} (AUC={auc:.3f})")
        plt.plot([0, 1], [0, 1], "--")
        plt.xlabel("False Positive Rate")
        plt.ylabel("True Positive Rate")
        plt.title(f"SimMIA ROC — BookTection ({LENGTH})")
        plt.legend(loc="lower right")
        plt.show()

        metrics_path = os.path.join(RESULTS_DIR, "simmia_metrics_summary.csv")

        metrics_summary = pd.DataFrame([{
            "method": METHOD,
            "model": MODEL,
            "backend": BACKEND,
            "length": LENGTH,
            "n_samples": len(sample_df),
            "max_words": MAX_WORDS,
            "n_samples_per_prefix": N_SAMPLES,
            "mean_absolute_error": sample_df["absolute_error"].mean(),
            "mean_error_members": sample_df[sample_df["label"] == 1]["absolute_error"].mean(),
            "mean_error_non_members": sample_df[sample_df["label"] == 0]["absolute_error"].mean(),
            "auc": auc,
            "auc_inverted": auc_inv,
            "n_non_member_shots": N_NON_MEMBER_SHOTS,
            "score_used_for_auc": "raw_membership_score",
            "score_used_for_mae": "membership_probability",
        }])

        metrics_summary.to_csv(metrics_path, index=False)

        print("\nMétricas guardadas en:")
        print(metrics_path)
    else:
        print("\nTodavía no se puede calcular AUC: falta una de las clases.")